In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install 'kaggle-environments>=0.1.6'

# Imports

In [ ]:
#Imports des bibliothèques

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# Préprocessing

In [ ]:
def preprocess_board (board, mark, config) :
    """
    Transforme la liste plate du plateau Kaggle en un tenseur multidimensionnel 
    optimisé pour un réseau de neurones convolutif (CNN).
    """  

    # Récupération des dimensions et conversion en matrice (Grid)
    rows, cols = config.rows, config.columns
    # On transforme la liste de 42 chiffres en une grille 2D de 6x7
    grid = np.array(board).reshape(rows, cols)
    # Si ma marque est 1, l'opposant est 2 (3-1=2). Si je suis 2, il est 1 (3-2=1).
    opp = 3 - mark

    # Création des 3 "canaux" (layers) 
    # Canal 1 : Où sont mes pions ? (1 si vrai, 0 si faux)
    layer_me = (grid == mark).astype(np.float32) 

    # Canal 2 : Où sont les pions adverses ?
    layer_opp = (grid == opp).astype(np.float32)

    # Canal 3 : Où sont les cases vides ? 
    layer_empty = (grid == 0).astype(np.float32)

    # Empilement des couches (Format CHW : Channels, Height, Width)
    # On crée un bloc de données de taille (3, 6, 7)
    stacked = np.stack([layer_me, layer_opp, layer_empty], axis=0) 

    # Conversion en Tenseur PyTorch et ajout de la dimension "Batch"
    return torch.tensor(stacked).unsqueeze(0)   # .unsqueeze(0) transforme (3, 6, 7) en (1, 3, 6, 7).

# Architecture CNN de valeur



In [ ]:
class ConnectXValueNet(nn.Module):
    """
    Réseau de neurones convolutif (CNN) de "Valeur". 
    Il prend un plateau en entrée et prédit la probabilité de victoire.
    """
    def __init__(self, rows, cols):
        super().__init__()

        # Première couche de convolution
        # Entre : 3 canaux (Moi, Opp, Vide). Sortie : 32 filtres différents.
        # kernel_size=4 permet de "voir" des alignements de 4 jetons d'un coup.
        self.conv1 = nn.Conv2d(3, 32, kernel_size=4, padding=1)
        
        # Deuxième couche de convolution
        # On affine la détection en combinant les motifs de la première couche.
        # Sortie : 64 filtres pour détecter des formes plus complexes (pièges, fourchettes).
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)

        # Couche de transition (Fully Connected)
        # On multiplie 64 (filtres) par la taille restante de la grille (rows-1 * cols-1)
        self.fc1 = nn.Linear(64 * (rows - 1) * (cols - 1), 128)
        
        # Couche de sortie finale
        # Un seul neurone qui donne la "note" finale du plateau.
        self.fc2 = nn.Linear(128, 1)

    def forward(self, x):
        # Activation ReLU : Introduit de la non-linéarité (ignore les signaux trop faibles).
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        
        # Aplatissage (Flattening) : On passe d'un bloc 3D (filtres) à un vecteur 1D.
        x = x.view(x.size(0), -1)
        
        # Analyse finale par les couches denses.
        x = F.relu(self.fc1(x))
        
        # Activation Tanh : borne le score entre -1 (défaite) et 1 (victoire).
        return torch.tanh(self.fc2(x))

# Coups valides

In [ ]:
def get_valid_moves(board, config):
    cols = config.columns
    #Calcul de l'indice de la colonne centrale
    center = cols // 2
    moves = [c for c in range(cols) if board[c] == 0]
    # On trie les colonnes selon leur distance par rapport au centre : On veut explorer d'abord le centre,
    return sorted(moves, key=lambda x: abs(x - center))

# Jouer un coup

In [ ]:
def play_move(board, col, mark, config):
    """
    Simule la chute d'un jeton dans une colonne et renvoie le nouveau plateau.
    """
    #Copie du plateau
    new_board = board.copy()
    rows, cols = config.rows, config.columns

    # On parcourt les lignes du bas vers le haut (reversed)
    for r in reversed(range(rows)):
        # Calcul de l'index dans la liste 1D (r * nombre_de_colonnes + index_colonne)
        idx = col + r*cols
        # On cherche la première case vide (0) en partant du bas
        if new_board[idx] == 0:
            new_board[idx] = mark
            # Une fois le jeton placé, on s'arrête (break)
            break

    return new_board


# Détection de victoire

In [ ]:
def check_winner(board, mark, config):
    """
    Vérifie si le joueur identifié par 'mark' a réussi un alignement gagnant.
    """
    # Extraction des paramètres et conversion en grille 2D pour la géométrie
    rows, cols, inarow = config.rows, config.columns, config.inarow
    grid = np.array(board).reshape(rows, cols)

    # Balayage systématique de tout le plateau
    for r in range(rows):
        for c in range(cols):
            # On ne lance les tests que si la case appartient au joueur
            if grid[r][c] != mark:
                continue

            # Exploration des 4 axes de victoire possibles à partir de cette case
            # (1,0) : Vertical | (0,1) : Horizontal | (1,1) : Diag Bas-Droite | (1,-1) : Diag Bas-Gauche
            for dr, dc in [(1,0), (0,1), (1,1), (1,-1)]:
                count = 0
                
                # Vérification de l'alignement sur la longueur requise (inarow = 4)
                for i in range(inarow):
                    # Calcul des coordonnées de la cellule suivante dans la direction choisie
                    rr, cc = r + dr*i, c + dc*i
                    
                    # Test de validité 
                    if 0 <= rr < rows and 0 <= cc < cols and grid[rr][cc] == mark:
                        count += 1
                    else:
                        break
                
                if count == inarow:
                    return True
    
    # Si après avoir tout fouillé, aucun alignement n'est trouvé
    return False

# Terminal

In [ ]:
def is_terminal(board, config):
    """
    Détermine si la partie est finie, soit par une victoire, soit par un match nul.
    """
    if check_winner(board, 1, config):
        return True

    if check_winner(board, 2, config):
        return True

    # Si toutes les cases sont remplies sans vainqueur, la partie s'arrête.
    if all(cell != 0 for cell in board):
        return True

    return False

# Evaluation de position

In [ ]:
def evaluate_position(board, mark, config, value_net):
    """
    Attribue un score à un plateau : score exact si la partie est finie, 
    ou estimation par le réseau de neurones sinon.
    """
    # Identification de l'adversaire
    opp = 3 - mark

    # Vérification des états terminaux 
    if check_winner(board, mark, config):
        return 1.0

    if check_winner(board, opp, config):
        return -1.0

    if all(cell != 0 for cell in board):
        return 0.0

    # Évaluation par l'IA pour les positions intermédiaires
    with torch.no_grad(): # 'torch.no_grad()' désactive le calcul des gradients pour accélérer l'inférence
        # On s'assure que les données sont sur le même matériel (CPU/GPU) que le réseau
        device = next(value_net.parameters()).device
        
        # Transformation du plateau en tenseur 
        x = preprocess_board(board, mark, config).to(device)
        
        # Le CNN prédit une valeur entre -1 et 1. 
        return value_net(x).item() # .item() transforme le tenseur PyTorch en un nombre flottant standard.

# Negamax + Alpha-Beta

In [ ]:
def negamax(board, depth, alpha, beta, mark, config, value_net):
    """
    Algorithme de recherche récursive avec élagage Alpha-Beta.
    Explore les futurs possibles pour maximiser le score du joueur actuel.
    """
    
    # Si on a atteint la profondeur max ou si la partie est finie
    if depth == 0 or is_terminal(board, config):
        # On retourne l'évaluation (Heuristique du CNN ou score exact)
        return evaluate_position(board, mark, config, value_net)

    best = -np.inf
    opp = 3 - mark

    # Exploration des coups possibles
    for col in get_valid_moves(board, config):
        # Simulation du coup par l'IA
        next_board = play_move(board, col, mark, config)
        
        # On inverse le score (-) car le gain de l'adversaire est notre perte
        score = -negamax(next_board, depth-1, -beta, -alpha, opp, config, value_net)

        # Mise à jour du meilleur score trouvé sur ce nœud
        best = max(best, score)
        alpha = max(alpha, score) # Alpha représente le meilleur score garanti

        # Élagage Alpha-Beta (Pruning)
        if alpha >= beta:
            break 

    return best        

# Choix du coup (avec heuristique puis minimax)

In [ ]:
def choose_best_move(board, mark, config, value_net, depth):
    """
    Détermine le meilleur coup à jouer en combinant heuristiques et une Negamax
    """
    opp = 3 - mark
    valid_moves = get_valid_moves(board, config)

    # Gagner immédiatement
    for col in valid_moves: 
        # Si jouer dans cette colonne complète un alignement de 4
        if check_winner(play_move(board, col, mark, config), mark, config):
            return col 

    # Bloquer l'adversaire
    for col in valid_moves:
        if check_winner(play_move(board, col, opp, config), opp, config):
            return col 

    # Lancement du Negamax : Si aucun coup n'est décisive, on lance la recherche
    best_score = -np.inf
    best_col = valid_moves[0] # Choix par défaut

    for col in valid_moves:
        # Simulation du coup
        next_board = play_move(board, col, mark, config)
        
        # On utilise -negamax car c'est au tour de l'adversaire (opp).
        score = -negamax(next_board, depth-1, -np.inf, np.inf, opp, config, value_net)

        # On garde le coup qui mène au meilleur score prédit
        if score > best_score:
            best_score = score
            best_col = col

    return best_col

# Agent final

In [ ]:
# Variable globale pour conserver le réseau en mémoire entre les tours
VALUE_NET = None 

def agent(observation, configuration):
    """
    Fonction principale appelée par Kaggle à chaque tour de jeu.
    """
    global VALUE_NET

    # On ne crée et ne charge le réseau qu'au tout premier tour pour gagner d temps
    if VALUE_NET is None: 
        # Création de l'architecture
        VALUE_NET = ConnectXValueNet(configuration.rows, configuration.columns)

        # Chargement des poids 
        load_weights(VALUE_NET)
        
        # Mode Évaluation 
        VALUE_NET.eval() 

    board = observation.board  # Liste de 42 entiers
    mark = observation.mark    # 1 ou 2 (mon numéro de joueur)

    # Gestion du temps, on peut pas aller trop en profondeur car kaggle impose 2sec par coup
    if observation.step < 4:
        # Début : Peu de pièces, on joue vite pour économiser 
        depth = 4
    elif observation.step < 14:
        # Milieu de jeu : Moment critique, on augmente la profondeur pour ne rater aucun piège
        depth = 6
    else:
        # Fin de jeu : On repasse à 5 car le plateau est encombré
        depth = 5

    # On lance la recherche du meilleur coup avec les paramètres actuels
    return int(choose_best_move(board, mark, configuration, VALUE_NET, depth))

# Self play

SELF-PLAY : génère positions -> ajoute symétrie -> buffer d'entraînement -> CNN apprend -> negamax utilise 

In [ ]:
def self_play_game(config, value_net, depth=4, epsilon=0.1):
    """
    Simule une partie complète de l'IA contre elle-même pour générer des données d'entraînement.
    Introduit de l'exploration (hasard) pour découvrir de nouvelles stratégies.
    """
    # Initialisation d'une nouvelle partie 
    board = [0] * (config.rows * config.columns)
    current_player = 1
    done = False

    # On stocke chaque état du plateau et le joueur qui devait agir
    game_history = []

    while not done:
        # Enregistrer la position avant que le coup ne soit joué
        game_history.append((board.copy(), current_player))

        # Durant les 8 premiers coups, on introduit une chance (epsilon) de jouer au hasard.
        if len(game_history) < 8 and np.random.rand() < epsilon:
            move = np.random.choice(get_valid_moves(board, config))
        else:
            # Sinon, l'IA joue son meilleur coup actuel 
            move = choose_best_move(board, current_player, config, value_net, depth)

        # Mise à jour du plateau
        board = play_move(board, move, current_player, config)

        # Conditions d'arrêt
        if check_winner(board, current_player, config):
            winner = current_player # Victoire
            done = True
        elif all(cell != 0 for cell in board):
            winner = 0 # Match nul
            done = True
        else:
            current_player = 3 - current_player # Changement de tour

    # Récompense et augmentation des données
    for past_board, past_player in game_history:
        # Attribution de la récompense (Value)
        if winner == 0:
            value = 0.0 # Match nul
        elif past_player == winner:
            value = 1.0 # Ce coup a mené à la victoire
        else:
            value = -1.0 # Ce coup a mené à la défaite

        # ajout du buffer : Stocke la position originale
        self_play_buffer.append((past_board, past_player, value))

        # symétrie : On multiplie les données par 2 (gauche/droite).
        grid = np.array(past_board).reshape(config.rows, config.columns)
        mirrored = np.fliplr(grid).flatten().tolist()
        self_play_buffer.append((mirrored, past_player, value))

# Entraînement du CNN via boucle

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np

class ConnectXDataset(Dataset):
    """
    Classe héritant de PyTorch Dataset. Elle organise les données récoltées (buffer) pour les rendre 
    utilisables par l'algorithme d'entraînement.
    """
    def __init__(self, buffer, config):
        self.boards = [] # Liste pour stocker les tenseurs de plateaux
        self.values = [] # Liste pour stocker les résultats (victoire/défaite)

        # Préparation et Nettoyage des données
        for board, mark, value in buffer:
            tensor = preprocess_board(board, mark, config) # On transforme chaque plateau en tenseur (3, 6, 7)
            
            # .squeeze(0) retire la dimension "batch" (1, 3, 6, 7) -> (3, 6, 7)
            self.boards.append(tensor.squeeze(0))
            self.values.append(value)

        # torch.stack transforme une liste de tenseurs en un seul gros bloc de données.
        self.boards = torch.stack(self.boards)
        
        # On transforme les scores en tenseur de flottants de forme (N, 1)
        # .unsqueeze(1) assure que la forme correspond exactement à la sortie du CNN.
        self.values = torch.tensor(self.values, dtype=torch.float32).unsqueeze(1)

    def __len__(self):
        """Retourne le nombre total d'exemples dans le dataset."""
        return len(self.boards)

    def __getitem__(self, idx):
        """
        Permet au DataLoader de récupérer une paire (Plateau, Valeur) à un index précis durant l'entraînement.
        """
        return self.boards[idx], self.values[idx]



In [ ]:
def train_value_net(value_net, buffer, config, epochs=3, batch_size=32, learning_rate=0.0005):
    """
    Lance la phase d'apprentissage : ajuste les poids du CNN pour que ses prédictions collent aux résultats réels observés dans le buffer.
    """

    # On transforme le buffer en Dataset, puis en DataLoader 
    dataset = ConnectXDataset(buffer, config)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    # Indique au réseau qu'il doit se préparer à mettre à jour ses neurones.
    value_net.train()

    # Choix de l'Optimiseur (Adam)
    # C'est l'algorithme qui décide de "combien" modifier chaque neurone. Adam est choisi pour son efficacité et sa gestion adaptative du learning rate.
    optimizer = torch.optim.Adam(value_net.parameters(), lr=learning_rate)

    # Fonction de Coût (MSE Loss)
    # On utilise l'Erreur Quadratique Moyenne car on fait de la régression : on veut que la prédiction soit la plus proche possible du réel 
    criterion = nn.MSELoss()

    # Boucle d'apprentissage (Epochs)
    for epoch in range(epochs):
        total_loss = 0.0

        for boards, values in dataloader:
            # Réinitialisation des gradients (on efface la mémoire du tour précédent)
            optimizer.zero_grad()

            # Le réseau fait une prédiction
            preds = value_net(boards)
            
            # calcul de l'érreur = Différence entre prédiction et réalité
            loss = criterion(preds, values)

            # On calcule l'erreur de chaque neurone
            loss.backward()
            
            # On ajuste les poids du réseau pour réduire l'erreur
            optimizer.step()

            # Cumul de la perte pour le suivi 
            total_loss += loss.item() * boards.size(0)

        # Calcul de la perte moyenne sur toute l'époque
        avg_loss = total_loss / len(dataset)
        print(f"Epoch {epoch+1}/{epochs} - Loss: {avg_loss:.4f}")

In [ ]:
#Préparation de lenvironnement
from kaggle_environments import make


env = make("connectx", debug=True)
config = env.configuration  # récupère les rows, columns, inarow exacts de Kaggle


In [ ]:
from kaggle_environments import evaluate

# Hyperparamètres
TOTAL_CYCLES = 5       # Nombre de fois où l'on répète tout le processus
GAMES_PER_CYCLE = 400  # Quantité de nouvelles données générées à chaque cycle
EPOCHS_PER_CYCLE = 2   # Entraînement court pour éviter le surapprentissage (overfitting)

best_score = -1.0   

# Initialisation
if VALUE_NET is None:
    print("Initialisation du réseau de neurones...")
    VALUE_NET = ConnectXValueNet(config.rows, config.columns)

# Boucle d'apprentissage
for cycle in range(TOTAL_CYCLES):
    print(f"\n=== Cycle {cycle+1}/{TOTAL_CYCLES} ===")

    # Calcul du epsilon
    # L'IA commence avec 15% de hasard et réduit ce bruit de 10% à chaque cycle. On garde un minimum de 2% 
    epsilon = max(0.02, 0.15 * (0.9 ** cycle))

    # On vide la mémoire pour ne s'entraîner que sur les parties les plus récentes
    self_play_buffer = []

    # L'IA joue contre elle-même pour découvrir de nouvelles situations
    for _ in range(GAMES_PER_CYCLE):
        self_play_game(config, VALUE_NET, depth=4, epsilon=epsilon)

    print("Positions collectées :", len(self_play_buffer))

    # On met à jour le cerveau avec les nouvelles parties collectées
    train_value_net(VALUE_NET, self_play_buffer, config, epochs=EPOCHS_PER_CYCLE )

    VALUE_NET.eval()

    # Test contre un joueur Aléatoire
    rewards_r = evaluate("connectx", [agent, "random"], num_episodes=100)
    mean_score_r = sum(r[0] for r in rewards_r) / len(rewards_r)
    
    # Test contre Negamax 
    rewards_n = evaluate("connectx", [agent, "negamax"], num_episodes=100)
    mean_score_n = sum(r[0] for r in rewards_n) / len(rewards_n)

    print(f"Bruit : {epsilon}")
    print(f"Score moyen vs Random  : {mean_score_r}")
    print(f"Score moyen vs Negamax : {mean_score_n}")

    # On ne sauvegarde les poids que si cette version est meilleure que la précédente contre Negamax
    if mean_score_n > best_score:
        best_score = mean_score_n
        torch.save(VALUE_NET.state_dict(), "value_net.pth")
        print("Meilleur CNN sauvegardé")

In [ ]:
import time

# On crée un plateau vide 
board = [0] * (config.rows * config.columns)
mark = 1

# time.time() donne l'heure précise en secondes
start = time.time()

# On lance la fonction principale avec une profondeur spécifique (depth=4)
choose_best_move(board, mark, config, VALUE_NET, depth=4)

# On soustrait l'heure de début à l'heure actuelle pour obtenir la durée
print("Temps coup :", time.time() - start)

# Visualisation du résultat

In [ ]:
VALUE_NET.eval()  # important pour inference rapide et stable


In [ ]:

# On crée l'environnement ConnectX (debug=True pour voir le plateau)
env = make("connectx", debug=True)


In [ ]:
#Test contre random

env.run([agent, "random"])

# Affiche le plateau final
env.render(mode="ipython", width=600, height=500, header=False)


In [ ]:
#Test contre negamax
env.run([agent, "negamax"])
env.render(mode="ipython", width=600, height=500, header=False)

# Submission

In [ ]:
import json

# On extrait les poids du réseau entraîné en format liste Python
trained_weights = {
    k: v.cpu().numpy().tolist()
    for k, v in VALUE_NET.state_dict().items()
}


In [ ]:
submission_code = r"""
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


def preprocess_board (board, mark, config) :

    rows, cols = config.rows, config.columns
    grid = np.array(board).reshape(rows, cols)
    opp = 3 - mark

    layer_me = (grid == mark).astype(np.float32) #True = 1, False = 0
    layer_opp = (grid == opp).astype(np.float32)
    layer_empty = (grid == 0).astype(np.float32)

    stacked = np.stack([layer_me, layer_opp, layer_empty], axis=0) #format CHW (Channels, Height, Width
    return torch.tensor(stacked).unsqueeze(0) 


class ConnectXValueNet(nn.Module):
    def __init__(self, rows, cols):
        super().__init__()

        self.conv1 = nn.Conv2d(3, 32, kernel_size=4, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)

        self.fc1 = nn.Linear(64 * (rows - 1) * (cols - 1), 128)
        self.fc2 = nn.Linear(128, 1)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        return torch.tanh(self.fc2(x))


def get_valid_moves(board, config):
    cols = config.columns
    center = cols // 2
    moves = [c for c in range(cols) if board[c] == 0]
    # ordre optimal pour alpha-beta
    return sorted(moves, key=lambda x: abs(x - center))


def play_move(board, col, mark, config):
    new_board = board.copy()
    rows, cols = config.rows, config.columns

    for r in reversed(range(rows)):
        idx = col + r*cols
        if new_board[idx] == 0:
            new_board[idx] = mark
            break

    return new_board


def check_winner(board, mark, config):
    rows, cols, inarow = config.rows, config.columns, config.inarow
    grid = np.array(board).reshape(rows, cols)

    for r in range(rows):
        for c in range(cols):
            if grid[r][c] != mark:
                continue

            for dr, dc in [(1,0),(0,1),(1,1),(1,-1)]:
                count = 0
                for i in range(inarow):
                    rr, cc = r + dr*i, c + dc*i
                    if 0 <= rr < rows and 0 <= cc < cols and grid[rr][cc] == mark:
                        count += 1
                if count == inarow:
                    return True
    return False


def is_terminal(board, config):

    if check_winner(board, 1,config):
        return True

    if check_winner(board, 2, config):
        return True

    if all(cell != 0 for cell in board):
        return True

    return False


def evaluate_position(board, mark, config, value_net):
    opp = 3-mark

    if check_winner(board, mark, config):
        return 1.0

    if check_winner(board, opp, config):
        return -1.0

    if all(cell !=0 for cell in board): #match nulle
        return 0.0

    with torch.no_grad(): #ne pas calculer les gradients = gain de temps
        x = preprocess_board(board, mark, config) #tenseur
        return value_net(x).item() #on envoie au reseau de neurone, item()=extrait le float du tenseur PyTorch


def negamax(board, depth, alpha, beta, mark, config, value_net):
    if depth == 0 or is_terminal(board, config):
        return evaluate_position(board, mark, config, value_net)

    best = -np.inf
    opp = 3 - mark

    for col in get_valid_moves(board, config):
        next_board = play_move(board, col, mark, config)
        score = -negamax(next_board, depth-1, -beta, -alpha, opp, config, value_net)

        best = max(best, score)
        alpha = max (alpha, score) #score du meilleur chemin trouvé jusqu'ici

        if alpha >= beta : #meilleur score trouvé par l'adversaire sur une autre branche
            break #alpha-beta pruning

    return best


def choose_best_move(board, mark, config, value_net, depth):

    opp = 3 - mark
    valid_moves = get_valid_moves(board, config)

    # Gagner immédiatement
    for col in valid_moves : 
        if check_winner(play_move(board, col, mark, config), mark, config):
            return col

    
    #Bloquer l'adversaire
    for col in valid_moves:
        if check_winner(play_move(board, col, mark, config), opp, config):
            return col

    #Minimax
    best_score = -np.inf
    best_col = valid_moves[0]

    for col in valid_moves:
        next_board = play_move(board, col, mark, config)
        score = -negamax(next_board, depth-1, -np.inf, np.inf, opp, config, value_net)

        if score > best_score:
            best_score = score
            best_col = col

    return best_col


def load_weights(model, weights_dict):
    state = {}
    for k, v in weights_dict.items():
        state[k] = torch.tensor(v, dtype=torch.float32)
    model.load_state_dict(state)

VALUE_NET = None

def agent(observation, configuration):
    global VALUE_NET
    if VALUE_NET is None:
        VALUE_NET = ConnectXValueNet(configuration.rows, configuration.columns)
        # On utilise les poids injectés dans le script
        load_weights(VALUE_NET, WEIGHTS)
        VALUE_NET.eval()

    board = observation.board
    mark = observation.mark
    depth = 5 if observation.step < 6 else 4
    return int(choose_best_move(board, mark, configuration, VALUE_NET, depth))



"""

with open("submission.py", "w") as f:
    f.write(f"WEIGHTS = {json.dumps(trained_weights)}\n")
    f.write(submission_code)


print("submission.py créé")

In [ ]:
#Vérification

from kaggle_environments import make

env = make("connectx", debug=True)

# Charger l'agent directement depuis le fichier
env.run(["submission.py", "submission.py"])

print("Status :", env.state[0].status, env.state[1].status)
